# Setup

In [ ]:
import os

import numpy as np

from fpu_util import fpuSplit, fpuMerge
from fft_util import FFT
from attack_util import (attack_sign, attack_mantissa, attack_exponent,
                         attack_two_exponent, check_mantissa, occurrence_prob)

size_f = 512
fft = FFT(size_f)

# Load Parameters and Correct Key

In [ ]:
B = 11              # Bound on |f1 / f2|
D = B * 64          # Filtering threshold on |c1 / c2|
f = np.load("f.npy")
F = fft.forward(f)

# Check Data

In [ ]:
# Inspect the stored hashes and the key at one target index.
target_idx = 1
c1_split_list, c2_split_list = [], []

re_hash_file = f"hashes/c_re_{D}x_im_idx{target_idx}.npy"
im_hash_file = f"hashes/c_im_{D}x_re_idx{target_idx}.npy"

if os.path.exists(re_hash_file):
    c_re_Dx_im_idx = np.load(re_hash_file)
    for c in c_re_Dx_im_idx:
        c1_split_list.append(list(fpuSplit(fft.forward(c)[target_idx].real)))
    c1_split_list = np.array(c1_split_list)
    print(f"Loaded hash file {re_hash_file}")
if os.path.exists(im_hash_file):
    c_im_Dx_re_idx = np.load(im_hash_file)
    for c in c_im_Dx_re_idx:
        c2_split_list.append(list(fpuSplit(fft.forward(c)[target_idx].imag)))
    c2_split_list = np.array(c2_split_list)
    print(f"Loaded hash file {im_hash_file}")

# Show the key and the first hash at target_idx.
f1, f2 = F[target_idx].real, F[target_idx].imag
ratio = np.abs(f1 / f2)
within = (1 / B) < ratio < B
print(f"Parameters: B = {B}, D = {D}, target_idx = {target_idx}")
print(f"{'F[idx]':<12} | {f1:<10.8g} | {f2:<10.8g}")
print(f"{'|f1/f2|':<12} | {ratio:10.8g} | is {'' if within else 'NOT '}within 1/{B} and {B}")
print("-" * 85)

for c in [c_re_Dx_im_idx[0], c_im_Dx_re_idx[0]]:
    C = fft.forward(c)
    c1, c2 = C[target_idx].real, C[target_idx].imag
    assert np.abs(c1 / c2) < 1 / D or np.abs(c1 / c2) > D
    print(f"{'C[idx]':<12} | {c1:<10.8g} | {c2:<15.10g}")
    complex_prod = f1 * c1 - f2 * c2
    s, e, m = fpuSplit(complex_prod)
    print(f"{'f1c1-f2c2':<12} | {s:<2} | {e:<5} | {bin(m + 2**52)}")
    s, e, m = fpuSplit(f1 * c1)
    print(f"{'f1c1':<12} | {s:<2} | {e:<5} | {bin(m + 2**52)}")
    print(f"{'f1c1/(prod)':<12} | {f1 * c1 / complex_prod:<10.8g}")
    s, e, m = fpuSplit(f2 * c2)
    print(f"{'f2c2':<12} | {s:<2} | {e:<5} | {bin(m + 2**52)}")
    print(f"{'f2c2/(prod)':<12} | {f2 * c2 / complex_prod:<10.8g}")
    print("-" * 85)

In [ ]:
# Load and plot stored traces for target_idx.
# The traces are recorded by ELMO; see gen_trace.ipynb.
re_trace_file = f"truncated_traces/traces_re_{D}x_im_idx{target_idx}.npy"
im_trace_file = f"truncated_traces/traces_im_{D}x_re_idx{target_idx}.npy"

if os.path.exists(re_trace_file):
    traces_re_Dx_im = np.load(re_trace_file)
    print(f"Trace file: {re_trace_file}")
    print(f"Loaded {traces_re_Dx_im.shape[0]} traces of length {traces_re_Dx_im.shape[1]}")
if os.path.exists(im_trace_file):
    traces_im_Dx_re = np.load(im_trace_file)
    print(f"Trace file: {im_trace_file}")
    print(f"Loaded {traces_im_Dx_re.shape[0]} traces of length {traces_im_Dx_re.shape[1]}")

In [ ]:
%matplotlib ipympl
import matplotlib.pyplot as plt


def plot_trace(trace):
    """Plot a single trace."""
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(trace, linewidth=0.7, color="tab:blue")
    ax.set_xlabel("Cycle number")
    ax.set_ylabel("Leakage value")
    ax.grid(True, alpha=0.3)
    plt.show()


plot_trace(traces_re_Dx_im[0])

# Attack

## Functions

In [ ]:
def load_split_hashes(hash_file, part, target_idx):
    """
    Load a hash file and return the (s, e, m) split of FFT(c)[target_idx].<part> per hash.

    Args:
        hash_file (str): Path to the .npy hash file.
        part (str): "real" or "imag".
        target_idx (int): FFT index to read.

    Returns:
        np.ndarray: Array of shape (n_hashes, 3) with the sign, exponent, and mantissa.
    """
    hashes = np.load(hash_file)
    split = [list(fpuSplit(getattr(fft.forward(c)[target_idx], part))) for c in hashes]
    return np.array(split)


def attack_idx(target_idx, D, highBits=8, trace_num=200, recover_exponent=True, hint=None):
    """
    Recover F[target_idx] = f1 + i*f2 from the stored hashes and traces.

    For each secret part, the sign and mantissa are recovered from the filtered traces,
    then the two exponents are recovered jointly from the unfiltered traces.

    Returns a dict with the per-part guesses and the underlying correlation results.
    """
    re_hash_file = f"hashes/c_re_{D}x_im_idx{target_idx}.npy"
    im_hash_file = f"hashes/c_im_{D}x_re_idx{target_idx}.npy"
    exist_re_Dx_im = os.path.exists(re_hash_file)
    exist_im_Dx_re = os.path.exists(im_hash_file)

    c1_split_list = load_split_hashes(re_hash_file, "real", target_idx) if exist_re_Dx_im else None
    c2_split_list = load_split_hashes(im_hash_file, "imag", target_idx) if exist_im_Dx_re else None

    # Load the filtered traces. They are recorded by ELMO; see gen_trace.ipynb.
    if exist_re_Dx_im:
        traces_re_Dx_im = np.load(f"truncated_traces/traces_re_{D}x_im_idx{target_idx}.npy")
        trace_length = traces_re_Dx_im.shape[1]
        print(f"Trace file: truncated_traces/traces_re_{D}x_im_idx{target_idx}.npy")
        print(f"Loaded {traces_re_Dx_im.shape[0]} traces of length {traces_re_Dx_im.shape[1]}")
    if exist_im_Dx_re:
        traces_im_Dx_re = np.load(f"truncated_traces/traces_im_{D}x_re_idx{target_idx}.npy")
        trace_length = traces_im_Dx_re.shape[1]
        print(f"Trace file: truncated_traces/traces_im_{D}x_re_idx{target_idx}.npy")
        print(f"Loaded {traces_im_Dx_re.shape[0]} traces of length {traces_im_Dx_re.shape[1]}")

    # When only one side is available, use all traces for it; otherwise split them.
    if exist_re_Dx_im != exist_im_Dx_re:
        trace_num_half = trace_num
    else:
        trace_num_half = trace_num // 2

    # Cycle indices of the two leakage points within the truncated traces, each a
    # single sample. These were found empirically by profiling the current
    # ELMO/FPR/FprMul.bin, and must be re-profiled if that binary changes (e.g. a
    # different SHAREN, compiler, or trace truncation shifts the leakage).
    T_REAL = 95   # cycle computing the real part  (f1c1 - f2c2)
    T_IMAG = 46   # cycle computing the imaginary part (f2c1 + f1c2)
    res = {}
    res["highBits"], res["trace_num"] = highBits, trace_num

    if exist_re_Dx_im:
        print("Recovering f1 using traces that compute f1c1 - f2c2")
        t_idx_start_11, t_idx_end_11 = T_REAL, T_REAL + 1
        res11_s = attack_sign(c1_split_list[:trace_num_half], traces_re_Dx_im[:trace_num_half], t_idx_start=t_idx_start_11, t_idx_end=t_idx_end_11)
        res11_m = attack_mantissa(c1_split_list[:trace_num_half], traces_re_Dx_im[:trace_num_half], highBits, t_idx_start=t_idx_start_11, t_idx_end=t_idx_end_11)
        mantissa = np.argmax(res11_m[0])
        res11_e = attack_exponent(c1_split_list[:trace_num_half], traces_re_Dx_im[:trace_num_half], mantissa, highBits, t_idx_start=t_idx_start_11, t_idx_end=t_idx_end_11)

        print("Recovering f2 using traces that compute f2c1 + f1c2")
        t_idx_start_21, t_idx_end_21 = T_IMAG, T_IMAG + 1
        res21_s = attack_sign(c1_split_list[:trace_num_half], traces_re_Dx_im[:trace_num_half], t_idx_start=t_idx_start_21, t_idx_end=t_idx_end_21)
        res21_m = attack_mantissa(c1_split_list[:trace_num_half], traces_re_Dx_im[:trace_num_half], highBits, t_idx_start=t_idx_start_21, t_idx_end=t_idx_end_21)
        mantissa = np.argmax(res21_m[0])
        res21_e = attack_exponent(c1_split_list[:trace_num_half], traces_re_Dx_im[:trace_num_half], mantissa, highBits, t_idx_start=t_idx_start_21, t_idx_end=t_idx_end_21)

        s11_guess, e11_guess, m11_guess = np.argmax(res11_s[0]), np.argmax(res11_e[0]), np.argmax(res11_m[0])
        s21_guess, e21_guess, m21_guess = np.argmax(res21_s[0]), np.argmax(res21_e[0]), np.argmax(res21_m[0])
        res["11"] = (res11_s, res11_e, res11_m)
        res["21"] = (res21_s, res21_e, res21_m)
    else:
        res11_s, res11_e, res11_m, res21_s, res21_e, res21_m = None, None, None, None, None, None
        s11_guess, e11_guess, m11_guess, s21_guess, e21_guess, m21_guess = None, None, None, None, None, None

    if exist_im_Dx_re:
        print("Recovering f1 using traces that compute f1c2 + f2c1")
        t_idx_start_12, t_idx_end_12 = T_IMAG, T_IMAG + 1
        res12_s = attack_sign(c2_split_list[:trace_num_half], traces_im_Dx_re[:trace_num_half], t_idx_start=t_idx_start_12, t_idx_end=t_idx_end_12)
        res12_m = attack_mantissa(c2_split_list[:trace_num_half], traces_im_Dx_re[:trace_num_half], highBits, t_idx_start=t_idx_start_12, t_idx_end=t_idx_end_12)
        mantissa = np.argmax(res12_m[0])
        res12_e = attack_exponent(c2_split_list[:trace_num_half], traces_im_Dx_re[:trace_num_half], mantissa, highBits, t_idx_start=t_idx_start_12, t_idx_end=t_idx_end_12)

        print("Recovering f2 using traces that compute -f2c2 + f1c1")
        t_idx_start_22, t_idx_end_22 = T_REAL, T_REAL + 1
        # The -f2c2 term flips the sign of c2, so attack with the sign-flipped split.
        rev_c2_split_list = c2_split_list.copy()
        for i in range(len(c2_split_list)):
            rev_c2_split_list[i][0] = 1 - rev_c2_split_list[i][0]
        res22_s = attack_sign(rev_c2_split_list[:trace_num_half], traces_im_Dx_re[:trace_num_half], t_idx_start=t_idx_start_22, t_idx_end=t_idx_end_22)
        res22_m = attack_mantissa(rev_c2_split_list[:trace_num_half], traces_im_Dx_re[:trace_num_half], highBits, t_idx_start=t_idx_start_22, t_idx_end=t_idx_end_22)
        mantissa = np.argmax(res22_m[0])
        res22_e = attack_exponent(rev_c2_split_list[:trace_num_half], traces_im_Dx_re[:trace_num_half], mantissa, highBits, t_idx_start=t_idx_start_22, t_idx_end=t_idx_end_22)

        s12_guess, e12_guess, m12_guess = np.argmax(res12_s[0]), np.argmax(res12_e[0]), np.argmax(res12_m[0])
        s22_guess, e22_guess, m22_guess = np.argmax(res22_s[0]), np.argmax(res22_e[0]), np.argmax(res22_m[0])
        res["12"] = (res12_s, res12_e, res12_m)
        res["22"] = (res22_s, res22_e, res22_m)
    else:
        res12_s, res12_e, res12_m, res22_s, res22_e, res22_m = None, None, None, None, None, None
        s12_guess, e12_guess, m12_guess, s22_guess, e22_guess, m22_guess = None, None, None, None, None, None

    def get_the_best_guess(resA, guessA, resB, guessB):
        """Return the (guess, correlation) with the higher correlation of the two sides."""
        if resA is None:
            return guessB, resB[0][guessB]
        if resB is None:
            return guessA, resA[0][guessA]
        return (guessA, resA[0][guessA]) if resA[0][guessA] > resB[0][guessB] else (guessB, resB[0][guessB])

    s1_guess, corr_s1 = get_the_best_guess(res11_s, s11_guess, res12_s, s12_guess)
    s2_guess, corr_s2 = get_the_best_guess(res21_s, s21_guess, res22_s, s22_guess)
    m1_guess, corr_m1 = get_the_best_guess(res11_m, m11_guess, res12_m, m12_guess)
    m2_guess, corr_m2 = get_the_best_guess(res21_m, m21_guess, res22_m, m22_guess)
    e1_guess, corr_e1 = get_the_best_guess(res11_e, e11_guess, res12_e, e12_guess)
    e2_guess, corr_e2 = get_the_best_guess(res21_e, e21_guess, res22_e, e22_guess)
    res["s1_guess"], res["corr_s1"] = s1_guess, corr_s1
    res["s2_guess"], res["corr_s2"] = s2_guess, corr_s2
    res["m1_guess"], res["corr_m1"] = m1_guess, corr_m1
    res["m2_guess"], res["corr_m2"] = m2_guess, corr_m2
    res["e1_guess_sole"], res["corr_e1_sole"] = e1_guess, corr_e1
    res["e2_guess_sole"], res["corr_e2_sole"] = e2_guess, corr_e2

    if recover_exponent is False:
        return res

    random_trace_file = f"truncated_traces/traces_idx{target_idx}.npy"
    if not os.path.exists(random_trace_file):
        raise FileNotFoundError(
            f"{random_trace_file} not found; generate it in gen_trace.ipynb or call with recover_exponent=False")
    traces = np.load(random_trace_file)
    print(f"Trace file: {random_trace_file}")
    print(f"Loaded {traces.shape[0]} traces of length {traces.shape[1]}")
    print("Recovering e1 and e2 using unfiltered traces")

    # Use the sign and mantissa recovered above.
    s1, m1 = s1_guess, m1_guess
    s2, m2 = s2_guess, m2_guess
    # Optionally override with ground-truth hints.
    if hint is not None:
        if hint["s1"] is not None:
            s1, s2 = hint["s1"], hint["s2"]
            print("Use correct sign")
        if hint["m1"] is not None:
            m1, m2 = hint["m1"], hint["m2"]
            print("Use correct mantissa")

    c_list = np.load("hashes/c_list.npy")
    C_list = [fft.forward(c)[target_idx] for c in c_list]
    res_e = attack_two_exponent(C_list[:trace_num], traces[:trace_num], s1, m1, s2, m2, highBits, t_idx_start=(T_REAL, T_IMAG), t_idx_end=(T_REAL+1, T_IMAG+1))
    res["e"] = res_e

    # Among the top pick_top correlations, keep those before the largest successive
    # drop, then pick the pair with the highest joint occurrence probability.
    pick_top = 5
    flat_arr = res_e[0].ravel()
    sorted_indices = np.argsort(flat_arr)[-pick_top:][::-1]  # Largest first.
    sorted_arr = flat_arr[sorted_indices]
    diffs = np.abs(np.diff(sorted_arr))
    max_gap_idx = np.argmax(diffs)
    top_indices = sorted_indices[:max_gap_idx + 1]
    rows, cols = np.unravel_index(top_indices, res_e[0].shape)
    prob_max = max(occurrence_prob[r + 1016] * occurrence_prob[c + 1016] / 10000 for r, c in zip(rows, cols))
    for r, c in zip(rows, cols):
        if occurrence_prob[r + 1016] * occurrence_prob[c + 1016] / 10000 == prob_max:
            e1_guess, e2_guess = r + 1016, c + 1016
            prob = occurrence_prob[e1_guess] * occurrence_prob[e2_guess] / 10000
            corr_e1e2 = res_e[0][e1_guess - 1016, e2_guess - 1016]
            res["e1_guess"], res["e2_guess"], res["corr_e1e2"], res["prob"] = e1_guess, e2_guess, corr_e1e2, prob
    return res


def attack_result(res, hint=None):
    """Reconstruct the complex guess f1_guess + i*f2_guess from a result dict."""
    e1_guess, e2_guess = res["e1_guess"], res["e2_guess"]
    s1_guess, m1_guess = res["s1_guess"], res["m1_guess"]
    s2_guess, m2_guess = res["s2_guess"], res["m2_guess"]
    highBits = res["highBits"]
    if hint is not None:
        if hint["s1"] is not None:
            s1_guess, s2_guess = hint["s1"], hint["s2"]
            print("Use correct sign")
        if hint["m1"] is not None:
            m1_guess, m2_guess = hint["m1"], hint["m2"]
            print("Use correct mantissa")
    f1_guess = fpuMerge(s1_guess, e1_guess, m1_guess * 2**(52 - highBits))
    f2_guess = fpuMerge(s2_guess, e2_guess, m2_guess * 2**(52 - highBits))
    return f1_guess + f2_guess * 1j

In [ ]:
def show_attack_result(f1, f2, res, display=True, recover_exponent=True):
    """
    Build (and optionally print) a per-part summary of the attack on F[idx] = f1 + i*f2.

    Only the recoverable part(s) are reported: both when |f1/f2| is within [1/B, B],
    otherwise only the dominant one.
    """
    highBits = res["highBits"]
    s1_guess, m1_guess = res["s1_guess"], res["m1_guess"]
    s2_guess, m2_guess = res["s2_guess"], res["m2_guess"]
    if recover_exponent:
        e1_guess, e2_guess = res["e1_guess"], res["e2_guess"]
    s1, e1, m1 = fpuSplit(f1)
    s2, e2, m2 = fpuSplit(f2)
    ratio = np.abs(f1 / f2)

    def mark(ok):
        return "recovered ✅" if ok else "not recovered ❌"

    def component(i, s, s_guess, m, m_guess, e, e_guess):
        lines = [f"sign bit s{i} {mark(s == s_guess)}",
                 f"mantissa m{i} {mark(check_mantissa(m, m_guess, highBits, -5) == 1)}"]
        if recover_exponent:
            lines.append(f"exponent e{i} {mark(e == e_guess)}")
        return "".join(line + "\n" for line in lines)

    e1g = e1_guess if recover_exponent else None
    e2g = e2_guess if recover_exponent else None

    info_all = ""
    info_all += f"f1 = {f1}\n"
    info_all += f"correct s1 = {s1:<12} | found s1 = {s1_guess:<5}\n"
    info_all += f"correct m1 = {(m1 + 2**52) >> (52 - highBits):<12} | found m1 = {m1_guess + 2**highBits:<5}\n"
    info_all += f"f2 = {f2}\n"
    info_all += f"correct s2 = {s2:<12} | found s2 = {s2_guess:<5}\n"
    info_all += f"correct m2 = {(m2 + 2**52) >> (52 - highBits):<12} | found m2 = {m2_guess + 2**highBits:<5}\n"
    if recover_exponent:
        info_all += f"correct e1,e2 = {e1:<4},{e2:<4} | found e1,e2 = {e1_guess:<4},{e2_guess:<4} | occur prob. {res['prob']:<.5g}\n"

    if (1 / B) < ratio < B:
        info_all += f"|f1/f2| = {ratio} is within 1/{B} and {B}\n"
        info_all += component(1, s1, s1_guess, m1, m1_guess, e1, e1g)
        info_all += component(2, s2, s2_guess, m2, m2_guess, e2, e2g)
    else:
        info_all += f"|f1/f2| = {ratio} is NOT within 1/{B} and {B}\n"
        if ratio <= (1 / B):
            info_all += component(2, s2, s2_guess, m2, m2_guess, e2, e2g)
        else:
            info_all += component(1, s1, s1_guess, m1, m1_guess, e1, e1g)

    if display:
        print(info_all)
    return info_all

## Attack

In [ ]:
highBits = 10
trace_num = 200
TARGET_INDICES = range(size_f // 2)
F_approx = np.zeros(size_f, dtype=complex)

for target_idx in TARGET_INDICES:
    # Skip indices whose traces have not been generated yet.
    if not os.path.exists(f"truncated_traces/traces_idx{target_idx}.npy"):
        continue
    f1, f2 = F[target_idx].real, F[target_idx].imag
    s1, e1, m1 = fpuSplit(f1)
    s2, e2, m2 = fpuSplit(f2)

    # For small indices, use the ground-truth sign bits for exponent recovery.
    hint = None
    if target_idx % 64 == 0 or target_idx % 64 == 63:
        hint = {"s1": s1, "m1": None, "s2": s2, "m2": None}
        # To also use the ground-truth mantissa for exponent recovery:
        # hint = {"s1": s1, "m1": m1 >> (52 - highBits), "s2": s2, "m2": m2 >> (52 - highBits)}

    res = attack_idx(target_idx, D, highBits, trace_num, recover_exponent=True, hint=hint)
    info = show_attack_result(f1, f2, res, display=True, recover_exponent=True)
    F_approx[target_idx] = attack_result(res, hint)
    F_approx[511 - target_idx] = np.conjugate(F_approx[target_idx])
    print(f"F[{target_idx}] = {F[target_idx]}, F_approx[{target_idx}] = {F_approx[target_idx]}")
    print("-" * 75)